In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import os
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

In [8]:
# ========================= CONFIG =========================
CHANNELS = list(range(41, 47))          # Start with 41-46
CONTAMINATION = 0.005
RANDOM_SEED = 42
N_ESTIMATORS = 200
MAX_SAMPLES = 256

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

In [9]:
# 1. PREPROCESS & TRAIN (cached)

cache_file = os.path.join(CACHE_DIR, "preprocessed_train_channels41-46.pkl")
if os.path.exists(cache_file):
    print("Loading cached train data...")
    data_scaled, df_train, _ = joblib.load(cache_file)
else:
    print("Preprocessing train.parquet...")
    df_train = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/train.parquet")
    channel_cols = [f'channel_{i}' for i in CHANNELS]
    df_train = df_train[['id'] + channel_cols + ['is_anomaly']].copy()
    df_train = df_train.sort_values('id').reset_index(drop=True)
    df_train[channel_cols] = df_train[channel_cols].ffill().bfill()
    data_scaled = df_train[channel_cols].values.copy()
    joblib.dump((data_scaled, df_train, None), cache_file)

Loading cached train data...


In [10]:
# Train model
model_file = os.path.join(CACHE_DIR, "isolation_forest_channels41-46.pkl")
if os.path.exists(model_file):
    model = joblib.load(model_file)
    print("Model loaded from cache.")
else:
    print("Training Isolation Forest...")
    model = IsolationForest(n_estimators=N_ESTIMATORS, max_samples=MAX_SAMPLES,
                            contamination=CONTAMINATION, random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(data_scaled)
    joblib.dump(model, model_file)
    print("Model trained and cached.")

Model loaded from cache.


In [ ]:
# Local proxy score on train (rough baseline)
print("\n=== Local Proxy Score on Train Set ===")
train_scores = model.decision_function(data_scaled)
train_pred = (model.predict(data_scaled) == -1).astype(int)
true_labels = df_train['is_anomaly'].values

prec = precision_score(true_labels, train_pred, zero_division=0)
rec = recall_score(true_labels, train_pred, zero_division=0)
f1 = f1_score(true_labels, train_pred, zero_division=0)

print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"Predicted anomaly rate: {train_pred.mean():.4%} | True rate: {true_labels.mean():.4%}")


=== Local Proxy Score on Train Set ===


In [ ]:
# 2. INFERENCE ON TEST
print("\nLoading test.parquet and generating submission...")
df_test = pd.read_parquet("/Users/alex/Downloads/esa-adb-challenge (1)/test.parquet")
channel_cols = [f'channel_{i}' for i in CHANNELS]
df_test = df_test[['id'] + channel_cols].copy()
df_test = df_test.sort_values('id').reset_index(drop=True)
df_test[channel_cols] = df_test[channel_cols].ffill().bfill()

test_data = df_test[channel_cols].values
test_pred = (model.predict(test_data) == -1).astype(int)

submission = pd.DataFrame({
    'id': df_test['id'],
    'anomaly_flag': test_pred
})

submission.to_parquet("submission_isoforest_channels41-46.parquet", index=False)
print(f"Submission saved! Shape: {submission.shape}")
print(f"Predicted anomalies in test: {test_pred.sum()} ({test_pred.mean():.4%})")
print("\nFirst 5 rows of submission:")
print(submission.head())

print("\n✅ Done! You can now upload 'submission_isoforest_channels41-46.parquet' as Late Submission on Kaggle to get the real score.")